In [12]:
#packages
import requests
import pandas as pd
import json
from glob import glob
from pathlib import Path
import csv
import numpy as np
import matplotlib.pyplot as plt
from censusdis import data
from censusdis.datasets import ACS5

# Tract-level weather summary dataset

One row per tract.

Returns: tract_weather


In [ ]:
#create a mapping: which tract gets which weather station name

# Socioeconomic dataset
One row per tract.

Returns: tract_socioeconomic


In [10]:
vars = [
    # Population
    "B01003_001E",
    # Income & Poverty
    "B19013_001E", "C17002_002E",
    # Housing & Vehicles
    "B25044_001E", "B25044_003E",
    # Race / Ethnicity
    "B02001_002E", "B02001_003E", "B02001_004E",
    "B02001_005E", "B02001_006E", "B02001_007E", "B02001_008E",
    # Housing Cost Burden
    "B25070_001E", "B25070_007E", "B25070_008E", "B25070_009E", "B25070_010E",
    # Education
    "B15003_022E",
    # Disability
    "B18135_001E",
    # Internet Access
    "B28002_002E",
    # Age 65+
    "B01001_020E",
    # Housing Tenure
    "B25003_003E"
]

#Pulling this data for all census tracts in NC

In [17]:
dataset = "acs/acs5"
vintage = 2022        #2024 comes out on Dec 11
state_fips = "37"     # North Carolina as string

df = data.download(
    dataset=dataset,
    vintage=vintage,
    download_variables=vars,
    # Geographic filters (must be strings)
    state=state_fips,
    county="*",        # all counties
    tract="*"          # all tracts within each county
)

#derived indicators

#Getting percentages
def pct(num, den):
    return (num/den).replace([pd.NA, float("inf")], pd.NA)

df["pct_below_50pct_poverty"] = pct(df["C17002_002E"], df["B01003_001E"])
df["pct_no_vehicle"] = pct(df["B25044_003E"], df["B25044_001E"])
df["pct_renters_cost_burdened"] = pct(
    df["B25070_007E"] + df["B25070_008E"] + df["B25070_009E"] + df["B25070_010E"],
    df["B25070_001E"]
)
df["pct_over65"] = pct(df["B01001_020E"], df["B01003_001E"])
df["pct_with_disability"] = pct(df["B18135_001E"], df["B01003_001E"])
#df["pct_no_internet"] = pct(df["B28002_002E"], df["B28002_001E"])
df["pct_nonwhite"] = 1 - pct(df["B02001_002E"], df[
    ["B02001_002E","B02001_003E","B02001_004E","B02001_005E",
     "B02001_006E","B02001_007E","B02001_008E"]
].sum(axis=1))
#df["pct_renters"] = pct(df["B25003_003E"], df["B25003_001E"])

#save dataframe
#df.to_csv("./data_processed/nc_census_tracts_acs5.csv", index=False)

'''
#downloading tract boundaries for mapping
df_geo = data.download(dataset=dataset, vintage=year,
                       variables=vars, geography="tract",
                       state=state, with_geometry=True)
df_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")
'''

'\n#downloading tract boundaries for mapping\ndf_geo = data.download(dataset=dataset, vintage=year,\n                       variables=vars, geography="tract",\n                       state=state, with_geometry=True)\ndf_geo.to_file("nc_census_tracts_acs5.geojson", driver="GeoJSON")\n'

In [18]:
rename_dict = {
    # Population
    "B01003_001E": "total_population",

    # Income & Poverty
    "B19013_001E": "median_household_income",
    "C17002_002E": "below_poverty_level",

    # Housing & Vehicles
    "B25044_001E": "total_households_vehicle_data",
    "B25044_003E": "households_no_vehicle",

    # Race / Ethnicity
    "B02001_002E": "white_alone",
    "B02001_003E": "black_alone",
    "B02001_004E": "american_indian_alaska_native",
    "B02001_005E": "asian_alone",
    "B02001_006E": "native_hawaiian_pacific_islander",
    "B02001_007E": "some_other_race",
    "B02001_008E": "two_or_more_races",

    # Housing Cost Burden (Gross Rent as % of Income)
    "B25070_001E": "total_renters",
    "B25070_007E": "rent_30_to_34_pct_income",
    "B25070_008E": "rent_35_to_39_pct_income",
    "B25070_009E": "rent_40_to_49_pct_income",
    "B25070_010E": "rent_50_plus_pct_income",

    # Education
    "B15003_022E": "bachelors_degree_or_higher",

    # Disability
    "B18135_001E": "total_with_disability",

    # Internet Access
    "B28002_002E": "broadband_subscription_any",

    # Age 65+
    "B01001_020E": "population_65_plus",

    # Housing Tenure
    "B25003_003E": "renter_occupied_housing_units",
}

df = df.rename(columns=rename_dict)


In [19]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df.drop(columns=['TRACT'])),
    columns=df.drop(columns=['TRACT']).columns,
    index=df.index
)

df_scaled['TRACT'] = df['TRACT']
df_scaled = df.drop(columns=["STATE", "COUNTY"])

#Columns scaled to have mean 0 and std of 1 except for the TRACT id
tract_socioeconomic = df_scaled

In [20]:
tract_socioeconomic.dtypes

TRACT                                object
total_population                      int64
median_household_income             float64
below_poverty_level                   int64
total_households_vehicle_data         int64
households_no_vehicle                 int64
white_alone                           int64
black_alone                           int64
american_indian_alaska_native         int64
asian_alone                           int64
native_hawaiian_pacific_islander      int64
some_other_race                       int64
two_or_more_races                     int64
total_renters                         int64
rent_30_to_34_pct_income              int64
rent_35_to_39_pct_income              int64
rent_40_to_49_pct_income              int64
rent_50_plus_pct_income               int64
bachelors_degree_or_higher            int64
total_with_disability                 int64
broadband_subscription_any            int64
population_65_plus                    int64
renter_occupied_housing_units   

# Outage-level dataset

Returns: outage_events

In [7]:
import re

outages_raw = pd.read_csv('/Users/JChuang/Documents/3S MP/MP/data_raw/outage_tracker.csv')

#Filter out North Carolina
outages_NC = outages_raw[outages_raw['state'] == "North Carolina"]
#Turn outage_start_estimate and outage_end_estimate to UTC datetime objects
outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
outages_NC['outage_end_estimate'] = pd.to_datetime(outages_NC['outage_end_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')

#Turn fix_duration_estimate into hours
def duration_to_hours(duration_str):
    if pd.isna(duration_str):
        return None
    match = re.match(r'(?:(\d+)d )?(?:(\d+)h )?(?:(\d+)m )?(?:(\d+)s)?', duration_str)
    if not match:
        return None
    days = int(match.group(1)) if match.group(1) else 0
    hours = int(match.group(2)) if match.group(2) else 0
    minutes = int(match.group(3)) if match.group(3) else 0
    seconds = int(match.group(4)) if match.group(4) else 0
    total_hours = days * 24 + hours + minutes / 60 + seconds / 3600
    return total_hours

outages_NC['fix_duration_hours'] = outages_NC['fix_duration_estimate'].apply(duration_to_hours)

outages_NC = outages_NC.sort_values("outage_start_estimate").reset_index(drop=True)


/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_13732/705568491.py:8: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_13732/705568491.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  outages_NC['outage_start_estimate'] = pd.to_datetime(outages_NC['outage_start_estimate'], utc=True).dt.tz_localize(None).dt.floor('S')
/var/folders/x4/39bns7ns63g9446c_4gzy_tr0000gn/T/ipykernel_13732/705568491.py:9: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.

In [8]:
outages_NC['block_fips'] = outages_NC['block_fips'].astype(str)

#Take out tract and county from block FIPS
outages_NC['TRACT'] = outages_NC['block_fips'].str[5:11]

outage_events = outages_NC

In [9]:
outage_events.dtypes

outage_identifer                  int64
device_lat                      float64
device_lon                      float64
block_fips                       object
convex_hull                      object
jurisdiction                     object
origin                           object
state                            object
county                           object
affected                          int64
cause                            object
outage_start_estimate    datetime64[ns]
outage_end_estimate      datetime64[ns]
fix_duration_estimate            object
outage_restored                  object
fix_duration_hours              float64
TRACT                            object
dtype: object

# Tract-level outage dataset

Returns: tract_outages


In [ ]:
#create customer minutes out
outage_events['customer_minutes_out'] = outage_events['fix_duration_hours'] * outage_events['affected'] * 60
#Add seasonal categories
outage_events['month'] = outage_events['outage_start_estimate'].dt.month
outage_events['is_summer'] = outage_events['month'].isin([6, 7, 8])
outage_events['is_winter'] = outage_events['month'].isin([12, 1, 2])


In [23]:
tract_outage_stats = outage_events.groupby("TRACT").agg(
    total_outages = ("outage_identifer", "count"),

    # Duration metrics
    mean_duration_hours = ("fix_duration_hours", "mean"),
    median_duration_hours = ("fix_duration_hours", "median"),
    max_duration_hours = ("fix_duration_hours", "max"),
    total_duration_hours = ("fix_duration_hours", "sum"),

    # Customers affected
    mean_customers_affected = ("affected", "mean"),
    max_customers_affected = ("affected", "max"),
    total_customers_affected = ("affected", "sum"),

    # CMO
    total_customer_minutes_out = ("customer_minutes_out", "sum"),

    # Seasonal
    pct_outages_in_summer = ("is_summer", "mean"),
    pct_outages_in_winter = ("is_winter", "mean"),
).reset_index()


In [24]:
tract_outages = tract_outage_stats
tract_outages.dtypes

TRACT                          object
total_outages                   int64
mean_duration_hours           float64
median_duration_hours         float64
max_duration_hours            float64
total_duration_hours          float64
mean_customers_affected       float64
max_customers_affected          int64
total_customers_affected        int64
total_customer_minutes_out    float64
pct_outages_in_summer         float64
pct_outages_in_winter         float64
dtype: object

# Save dataframes to csvs for later use